# 🦙 Local Text-to-SQL RAG (Ollama & Qwen 2.5)

This notebook demonstrates how to query massive tabular datasets instantly using **DuckDB** and **LangChain**.

We are using **Ollama** to run the AI completely offline on your personal computer! This means **ZERO** rate limits, infinite requests, and completely private data processing.

### Step 1: Install Dependencies

In [1]:
# Install the necessary libraries for our pipeline
#!pip install -q duckdb duckdb-engine langchain langchain-classic langchain-community langchain-ollama sqlalchemy==2.0.44

### Step 2: Build the Database instantly with DuckDB
Instead of loading 1.4M rows into memory (RAM) with Pandas which causes crashes, we build a lightning-fast local analytical database.

In [2]:
import duckdb
import os
import pandas as pd

# File paths
db_path = "apple_sales_rag_ollama.db"
csv_path = "../data/processed/cleaned_apple_sales_v3.csv"

# Force-close any existing DuckDB connections and remove stale db file
try:
    duckdb.close()  # close default connection if open
except: pass
if os.path.exists(db_path):
    try: os.remove(db_path)
    except OSError:
        # If file is locked, use a fresh name instead
        import time
        db_path = f"apple_sales_rag_{int(time.time())}.db"
        print(f"\u26a0 Previous DB locked, using: {db_path}")

print(f"Connecting to DuckDB and loading dataset from {csv_path}...")
con = duckdb.connect(db_path)
con.execute(f"CREATE TABLE sales AS SELECT * FROM read_csv_auto('{csv_path}')")
print("\u2705 Successfully loaded 1 Million rows into DuckDB!")

print("\nSchema (What the Local LLM sees):")
display(con.execute("DESCRIBE sales").df())
con.close()


Connecting to DuckDB and loading dataset from ../data/processed/cleaned_apple_sales_v3.csv...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Successfully loaded 1 Million rows into DuckDB!

Schema (What the Local LLM sees):


,column_name,column_type,null,key,default,extra
0,sale_id,VARCHAR,YES,None,None,None
1,sale_date,DATE,YES,None,None,None
2,store_id,VARCHAR,YES,None,None,None
3,product_id,VARCHAR,YES,None,None,None
4,quantity,BIGINT,YES,None,None,None
5,product_name,VARCHAR,YES,None,None,None
6,launch_date,DATE,YES,None,None,None
7,price,DOUBLE,YES,None,None,None
8,store_name,VARCHAR,YES,None,None,None
9,city,VARCHAR,YES,None,None,None


### Step 3: Advanced AI Prompt Engineering
Because local models don't naturally understand the context of your data, we inject a **Custom System Prompt**. 
This acts as the "brain" or instruction manual for the AI, giving it custom logic hooks for the Apple Retail dataset.

In [3]:
from langchain_community.utilities import SQLDatabase
from langchain_classic.chains import create_sql_query_chain
from langchain_ollama import ChatOllama
from langchain_community.tools import QuerySQLDatabaseTool
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Connect LangChain to DuckDB
db = SQLDatabase.from_uri(f"duckdb:///{db_path}")

# 2. Initialize Qwen2.5 (temperature=0 for SQL, slightly warm for analysis)
llm = ChatOllama(model="qwen2.5-coder:3b", temperature=0)
analyst_llm = ChatOllama(model="qwen2.5-coder:3b", temperature=0.3)

# 3. SQL Generation Prompt — 15 strict business rules for accurate query generation
custom_prompt = PromptTemplate.from_template(
    """You are an elite DuckDB SQL programming assistant answering questions about Apple Retail Sales data.
    Given an input question, create a syntactically correct DuckDB query to run.
    
    Never query for all the columns from a specific table, only ask for the few relevant columns given the question.
    Be careful to not query for columns that do not exist.
    
    CRITICAL APPLE RETAIL BUSINESS RULES:
    1. If asked about "Sales", "Revenue", or "Income", use the 'sales_amount_realistic' column.
    2. If asked about Volume or Item Counts, use 'quantity_realistic'.
    3. If asked about a Country, filter using 'country_norm_mapped'.
    4. There is ONLY ONE table named 'sales'. DO NOT JOIN other tables.
    5. If asked to count "transactions" or "orders", use COUNT(*).
    6. If comparing metrics across different years, use conditional aggregation: SUM(CASE WHEN year=2023 THEN column END) AS year_2023, SUM(CASE WHEN year=2024 THEN column END) AS year_2024.
    7. Output ONLY the raw SQL string. No markdown, no explanation.
    8. To find the "most" or "least", use ORDER BY + LIMIT 1, never MAX()/MIN() with unaggregated columns.
    9. When asked for the PRICE of a product, ALWAYS use 'price_realistic'. NEVER use 'sales_amount_realistic' for prices.
    10. When filtering for a specific product line, use product_name LIKE '%ProductName%'.
    
    ADDITIONAL CRITICAL RULES:
    11. When asked about GDP or GDP per capita, ALWAYS use the 'gdp_per_capita' column. GDP is NOT revenue.
    12. When asked about a SPECIFIC product (e.g. 'iPhone 14'), filter EXACTLY for that product. For example: product_name LIKE '%iPhone 14%'. Do NOT use a broader filter like '%iPhone%' unless the user asks about ALL iPhones.
    13. When asked 'which products can I afford' or about a budget, ALWAYS: (a) use AVG(price_realistic) grouped by product_name, (b) filter with HAVING AVG(price_realistic) <= budget, (c) ONLY include the product category the user asked about (e.g. if they ask about iPhones, add WHERE product_name LIKE '%iPhone%').
    14. When comparing exactly 2 cities or locations, ALWAYS add WHERE city IN ('City1', 'City2') to filter only those cities.
    15. When asked if a price 'dropped' or 'changed' between years, use conditional aggregation to get separate values per year, e.g.: AVG(CASE WHEN year=2023 THEN price_realistic END) AS price_2023, AVG(CASE WHEN year=2024 THEN price_realistic END) AS price_2024.
    
    Only use the following tables:
    {table_info}

    Return a maximum of {top_k} results unless otherwise specified.

    Question: {input}"""
)

# 4. CEO Analyst Assistant Prompt — converts raw SQL results into executive insights
analyst_prompt = ChatPromptTemplate.from_template(
    """You are a Senior Data Analyst presenting findings to Apple's executive leadership.

Your communication style:
- Professional, confident, and concise
- Lead with the key insight first, then supporting details
- Use exact numbers with proper formatting (commas, currency symbols, percentages)
- Add brief business context or actionable takeaways when relevant
- If the data shows a trend, highlight it
- Keep responses to 2-4 sentences for simple queries, up to a short paragraph for complex ones
- Never mention SQL, databases, tables, columns, or technical implementation details
- If the result is empty or an error, say so clearly and suggest why

The user asked: "{question}"

The data returned: {result}

Provide your executive briefing:"""
)

# 5. Build the chains
write_query = create_sql_query_chain(llm, db, prompt=custom_prompt)
execute_query = QuerySQLDatabaseTool(db=db)
analyst_chain = analyst_prompt | analyst_llm | StrOutputParser()

print("\u2705 SQL Engine + CEO Analyst Assistant are ready!")
print("   \u2699  SQL Generation: Qwen 2.5 (temperature=0, 15 business rules)")
print("   \U0001f4ca Analyst Persona: Qwen 2.5 (temperature=0.3)")


✅ SQL Engine + CEO Analyst Assistant are ready!
   ⚙  SQL Generation: Qwen 2.5 (temperature=0, 15 business rules)
   📊 Analyst Persona: Qwen 2.5 (temperature=0.3)


d:\anaconda\envs\Apple\Lib\site-packages\duckdb_engine\__init__.py:184: DuckDBEngineWarning: duckdb-engine doesn't yet support reflection on indices
  warnings.warn(


### Step 4: The 100% Offline SQL Gauntlet!
Let's test both simple questions and extremely hard Data Science database queries.

In [4]:
def ask_local_ai(question):
    """Ask the local AI a question and get an executive-style briefing."""
    print(f"\n{'='*60}")
    print(f"\U0001f4ac  {question}")
    print(f"{'='*60}")
    
    # Step 1: Generate SQL
    sql_query = write_query.invoke({"question": question})
    clean_sql = sql_query.replace("```sql", "").replace("```", "").replace("SQLQuery:", "").strip()
    
    # Safety: inject LIMIT if not present
    if "LIMIT" not in clean_sql.upper():
        clean_sql = clean_sql.rstrip(";") + " LIMIT 50;"
    
    print(f"\n\u2699  SQL: {clean_sql}")
    
    # Step 2: Execute SQL
    try:
        raw_result = execute_query.invoke(clean_sql)
    except Exception as e:
        print(f"\n\u274c  Query failed: {e}")
        return
    
    # Step 3: Generate executive briefing via analyst LLM
    print(f"\n\U0001f50d  Raw Data: {str(raw_result)[:200]}{'...' if len(str(raw_result)) > 200 else ''}")
    
    try:
        briefing = analyst_chain.invoke({
            "question": question,
            "result": raw_result
        })
        print(f"\n\U0001f4ca  Analyst Briefing:")
        print(f"   {briefing}")
    except Exception as e:
        # Fallback to raw result if analyst LLM fails
        print(f"\n\U0001f4cb  Result: {raw_result}")
    
    print()


#### Easy Level Tests (Standard Analytics)

In [5]:
ask_local_ai("How many unique stores do we have in our entire dataset?")
ask_local_ai("What are the distinct product categories we sell? List them out.")
ask_local_ai("Which country had the highest number of overall sales transactions (not volume, just number of rows)?")


💬  How many unique stores do we have in our entire dataset?

⚙  SQL: SELECT COUNT(DISTINCT store_id) AS unique_stores FROM sales LIMIT 50;

🔍  Raw Data: [(75,)]

📊  Analyst Briefing:
   **Executive Briefing**

Good morning, Apple's leadership team.

Today, I'm presenting a key insight from our latest analysis. We have 75 unique stores across our dataset.

This is a significant number, indicating the breadth of our retail presence globally. Each store contributes to our diverse customer base and geographical reach.

Moving forward, it's crucial to monitor these numbers closely as they can influence strategic decisions related to expansion or market optimization.

Thank you for your attention.


💬  What are the distinct product categories we sell? List them out.

⚙  SQL: SELECT DISTINCT category_name FROM sales LIMIT 50;

🔍  Raw Data: [('Accessories',), ('Audio',), ('Desktop',), ('Smartphone',), ('Tablet',), ('Wearable',), ('Streaming Device',), ('Subscription Service',), ('Laptop',), (

#### Medium Level Tests (Mathematical Inference & Data rules)

In [6]:
ask_local_ai("Which country sold the absolute most physical items (volume) out of all the countries combined?")
ask_local_ai("What is the average Apple revenue for the iPhone 14 in Japan in 2024? Remember to use the realistic amount.")



💬  Which country sold the absolute most physical items (volume) out of all the countries combined?

⚙  SQL: SELECT country_norm_mapped, SUM(quantity_realistic) AS total_volume
FROM sales
GROUP BY country_norm_mapped
ORDER BY total_volume DESC
LIMIT 1;

🔍  Raw Data: [('united states', 1663887)]

📊  Analyst Briefing:
   **Executive Briefing**

**Key Insight:** The United States sold the absolute most physical items (volume) out of all countries combined, with a total volume of **1,663,887 units**.

**Business Context:** This significant sales volume underscores the US's strong presence in various markets and its role as a global leader in consumer goods. Understanding this trend can inform Apple’s strategic decisions regarding market expansion, product development, and supply chain management.

**Actionable Takeaway:** Apple should focus on maintaining and expanding its market share in the United States while also exploring opportunities to leverage its brand influence across other regi

#### Advanced Level Tests (HAVING clauses & Time Series Grouping)

In [ ]:
ask_local_ai("Show me the top 3 stores with the highest average promo_flag impact, but filter out any store with less than 1000 total sales transactions.")



💬  Show me the top 3 stores with the highest average promo_flag impact, but filter out any store with less than 1000 total sales transactions.

⚙  SQL: SELECT store_name, AVG(promo_flag) AS avg_promo_flag_impact
FROM sales
GROUP BY store_name
HAVING COUNT(*) >= 1000
ORDER BY avg_promo_flag_impact DESC
LIMIT 3;

🔍  Raw Data: [('Apple Kaerntner Strasse', 0.16422764227642275), ('Apple Jewel Changi Airport', 0.1640890491712324), ('Apple Walnut Street', 0.16396833321446402)]

📊  Analyst Briefing:
   ### Executive Briefing

**Key Insight:** The top three stores with the highest average promo_flag impact are Apple Kaerntner Strasse, Apple Jewel Changi Airport, and Apple Walnut Street. Each of these stores has an average promo_flag impact of approximately 16.4%. This indicates a significant positive influence on sales transactions.

### Supporting Details:

- **Apple Kaerntner Strasse:** With an average promo_flag impact of 0.1642, this store stands out as the most impactful in terms of promo

In [8]:
ask_local_ai("top 5 stores")


💬  top 5 stores

⚙  SQL: SELECT store_name, COUNT(*) AS transaction_count
FROM sales
GROUP BY store_name
ORDER BY transaction_count DESC
LIMIT 5;

🔍  Raw Data: [('Apple Chadstone', 28621), ('Apple Covent Garden', 28521), ('Apple The Dubai Mall', 28478), ('Apple Central World', 28294), ('Apple Orchard Road', 28196)]

📊  Analyst Briefing:
   Executive Briefing:

The top five stores for Apple sales, based on the data provided, are as follows:

1. **Apple Chadstone** - Sales: $286,210
2. **Apple Covent Garden** - Sales: $285,210
3. **Apple The Dubai Mall** - Sales: $284,780
4. **Apple Central World** - Sales: $282,940
5. **Apple Orchard Road** - Sales: $281,960

These stores collectively account for a significant portion of Apple's total sales, demonstrating their strong presence in these locations.

### Actionable Takeaways:
- **Investment Focus**: Consider increasing investments in these top five stores to capitalize on their high sales performance.
- **Market Expansion**: Analyze the t

In [9]:
ask_local_ai("top 10 products")


💬  top 10 products

⚙  SQL: SELECT product_name, SUM(quantity_realistic) AS total_quantity_sold
FROM sales
GROUP BY product_name
ORDER BY total_quantity_sold DESC
LIMIT 10;

🔍  Raw Data: [('Apple Music', 237633), ('Apple Arcade', 228607), ('Apple News+', 227633), ('Apple TV+', 217388), ('Apple Watch Series 10', 207095), ('iPhone 16 Plus', 204121), ('iPhone 16e', 201829), ('iPhone 17',...

📊  Analyst Briefing:
   **Executive Briefing**

Dear Apple Executives,

I am pleased to present our latest data analysis on the top 10 products across our portfolio. The findings reveal a clear trend in consumer preferences, with several high-growth titles leading the way.

1. **Apple Music**: With over 237,633 units sold, Apple Music continues to be the top-selling product, demonstrating its strong appeal and versatility.
   
2. **Apple Arcade**: This subscription-based service has also shown significant growth, with 228,607 units sold, underscoring its growing popularity among consumers.

3. **Appl

In [10]:
ask_local_ai("top city")


💬  top city

⚙  SQL: SELECT city FROM sales GROUP BY city ORDER BY SUM(quantity_realistic) DESC LIMIT 1;

🔍  Raw Data: [('London',)]

📊  Analyst Briefing:
   **Executive Briefing on Top City**

The top city in our dataset is **London**, with a population of 9 million. This makes London the most populous city in the UK, which is significant given its historical and economic importance to both the United Kingdom and the global economy.

**Actionable Takeaway:** Given London's size and economic significance, it would be beneficial for Apple to explore opportunities related to urban development, technology integration, or strategic partnerships within the city. This could include investments in public transportation systems, innovation hubs, or collaborations with local businesses to enhance London’s competitiveness and quality of life.

**Business Context:** The top city is crucial for Apple's global strategy as it represents a major market for its products and services. Understanding tr

In [11]:
ask_local_ai("how many columns do we have not from sales")


💬  how many columns do we have not from sales

⚙  SQL: SELECT count(*) FROM information_schema.columns WHERE table_name = 'sales' LIMIT 50;

🔍  Raw Data: [(35,)]

📊  Analyst Briefing:
   **Executive Briefing**

Dear Apple Executives,

Thank you for scheduling this meeting. I am pleased to present our findings on the number of columns in our dataset that are not related to sales.

Based on the analysis, we have identified 35 columns that do not originate from the sales data. This indicates a clear separation between operational and financial metrics within our database.

**Actionable Takeaways:**

1. **Data Segmentation:** Understanding this segmentation can help us more effectively manage and analyze different types of data.
2. **Resource Allocation:** It may be beneficial to allocate additional resources for further analysis on these non-sales columns, potentially uncovering valuable insights that could enhance decision-making processes.

Thank you again for your attention to this ma

In [12]:
ask_local_ai("How many total columns are there in the sales table?")


💬  How many total columns are there in the sales table?

⚙  SQL: SELECT count(*) FROM information_schema.columns WHERE table_name = 'sales' LIMIT 50;

🔍  Raw Data: [(35,)]

📊  Analyst Briefing:
   Executive Briefing:

**Key Insight:** There are 35 total columns in the sales table.

**Supporting Details:** The sales table includes essential fields such as product ID, customer name, sale date, price per unit, quantity sold, and total amount. This comprehensive structure allows for detailed analysis of sales data across various dimensions.

**Business Context:** Understanding the number of columns helps in planning future enhancements to the database schema, ensuring that all necessary information is captured efficiently. The clarity of column names also aids in query optimization and user experience.

**Actionable Takeaways:**
- **Enhance Data Analysis Tools:** With 35 columns, advanced data analysis tools can be effectively utilized for complex queries and trend identification.
- **Opt

In [13]:
ask_local_ai("how many rows do we have")


💬  how many rows do we have

⚙  SQL: SELECT COUNT(*) FROM sales LIMIT 50;

🔍  Raw Data: [(1068918,)]

📊  Analyst Briefing:
   **Executive Briefing**

**Key Insight:** We have a total of 1,068,918 rows in our dataset.

**Supporting Details:** This figure represents the comprehensive count of records across all tables and datasets within our system. It includes all entries, regardless of their status (active, inactive, etc.), providing a full overview of our data volume.

**Business Context:** The total number of rows is crucial for understanding the scale of our data assets. This information helps in planning storage needs, optimizing query performance, and ensuring that we have enough resources to handle future growth without compromising on efficiency or accuracy.

**Actionable Takeaways:**

1. **Data Management:** Ensure that all systems are aligned with this total count to maintain consistency and avoid any discrepancies.
2. **Resource Allocation:** Review current resource allocati

In [14]:
# GDP Query — Rule 11 ensures the LLM uses gdp_per_capita, not revenue
ask_local_ai("What is the average GDP per capita for Japan by year? Use the gdp_per_capita column.")



💬  What is the average GDP per capita for Japan by year? Use the gdp_per_capita column.

⚙  SQL: SELECT year, AVG(gdp_per_capita) AS avg_gdp_per_capita
FROM sales
WHERE country_norm_mapped = 'japan'
GROUP BY year LIMIT 50;

🔍  Raw Data: [(2021, 40028.73417000103), (2022, 40094.55997999893), (2023, 34065.643900002404), (2024, 33836.17563000044), (2025, 32475.89250000095)]

📊  Analyst Briefing:
   **Executive Briefing**

The average GDP per capita for Japan by year is as follows:

- **2021**: ¥40,028.73 (approximately $3,638)
- **2022**: ¥40,094.56 (approximately $3,647)
- **2023**: ¥34,065.64 (approximately $2,912)
- **2024**: ¥33,836.18 (approximately $2,850)
- **2025**: ¥32,475.90 (approximately $2,718)

**Trend Analysis:**
There has been a significant decrease in Japan's average GDP per capita from 2021 to 2023, followed by a slight increase in 2024 and then another decline in 2025. This trend could be attributed to various factors such as economic fluctuations, changes in governmen

In [15]:
# iPhone 13 & 14 Price — Rule 12 ensures exact product filtering
ask_local_ai("What is the average price_realistic of the iPhone 13 and iPhone 14 by year? Filter using product_name LIKE '%iPhone 13%' OR product_name LIKE '%iPhone 14%'. Do NOT include other iPhones.")



💬  What is the average price_realistic of the iPhone 13 and iPhone 14 by year? Filter using product_name LIKE '%iPhone 13%' OR product_name LIKE '%iPhone 14%'. Do NOT include other iPhones.

⚙  SQL: SELECT 
    AVG(price_realistic) AS average_price,
    year
FROM 
    sales
WHERE 
    product_name LIKE '%iPhone 13%' OR product_name LIKE '%iPhone 14%'
GROUP BY 
    year LIMIT 50;

🔍  Raw Data: [(890.5540835792785, 2021), (926.6898089780601, 2022), (882.2004677007203, 2023), (816.2416415493935, 2024), (724.551705884302, 2025)]

📊  Analyst Briefing:
   **Executive Briefing**

**Key Insight:** The average price_realistic of the iPhone 13 and iPhone 14 by year has shown a consistent upward trend since 2021.

**Supporting Details:**
- **iPhone 13:** 
  - In 2021, the average price was $890.55.
  - By 2022, it increased to $926.69.
  - In 2023, it further rose to $882.20.
  - For 2024, the average price dropped slightly to $816.24.
  - By 2025, it stabilized at $724.55.

**Business Context:*

In [16]:
# Budget Query — Rule 13 ensures iPhone-only filtering + AVG price
ask_local_ai("Which iPhone models have an average price_realistic under $1000? Show product_name and AVG(price_realistic), filter WHERE product_name LIKE '%iPhone%', GROUP BY product_name, HAVING AVG(price_realistic) <= 1000, ORDER BY AVG(price_realistic).")



💬  Which iPhone models have an average price_realistic under $1000? Show product_name and AVG(price_realistic), filter WHERE product_name LIKE '%iPhone%', GROUP BY product_name, HAVING AVG(price_realistic) <= 1000, ORDER BY AVG(price_realistic).

⚙  SQL: SELECT product_name, AVG(price_realistic) AS avg_price
FROM sales
WHERE product_name LIKE '%iPhone%'
GROUP BY product_name
HAVING AVG(price_realistic) <= 1000
ORDER BY avg_price LIMIT 50;

🔍  Raw Data: [('iPhone SE (2nd Gen)', 356.834641027497), ('iPhone SE (3rd Gen)', 422.14109421217506), ('iPhone 11', 576.9902469318575), ('iPhone 16e', 603.6950065943842), ('iPhone 12 mini', 624.3616718220376), ('i...

📊  Analyst Briefing:
   Based on the data provided, here are the iPhone models with an average price_realistic under $1000:

- **iPhone SE (2nd Gen)**: Average price is $356.83
- **iPhone SE (3rd Gen)**: Average price is $422.14
- **iPhone 11**: Average price is $576.99
- **iPhone 12 mini**: Average price is $624.36
- **iPhone 13 mini*

In [17]:
# Affordability Query — Rule 13 ensures aggregated price comparison
ask_local_ai("What is the average price_realistic of the iPhone 14 and the average price_realistic of any iPad? Show product_name and AVG(price_realistic). Filter WHERE product_name LIKE '%iPhone 14%' OR product_name LIKE '%iPad%'. GROUP BY product_name.")



💬  What is the average price_realistic of the iPhone 14 and the average price_realistic of any iPad? Show product_name and AVG(price_realistic). Filter WHERE product_name LIKE '%iPhone 14%' OR product_name LIKE '%iPad%'. GROUP BY product_name.

⚙  SQL: SELECT product_name, AVG(price_realistic) AS avg_price_realistic
FROM sales
WHERE product_name LIKE '%iPhone 14%' OR product_name LIKE '%iPad%'
GROUP BY product_name LIMIT 50;

🔍  Raw Data: [('iPhone 14 Plus', 885.5952315256492), ('iPad Pro 11-inch (3rd Gen)', 771.0208910286146), ('iPad mini (5th Gen)', 329.03287011749995), ('iPhone 14 Pro', 983.9015251663517), ('iPhone 14', 786.24756763...

📊  Analyst Briefing:
   **Executive Briefing**

The average price_realistic for the iPhone 14 is **$786.25**, and for any iPad, it is **$441.50**.

**Key Insights:**
- The iPhone 14 Plus has the highest average price_realistic at **$885.59**.
- The iPad Pro 11-inch (3rd Gen) follows with an average price_realistic of **$771.02**.
- The iPad mini (5t

In [18]:
# City Price Comparison — Rule 14 ensures only 2 cities are compared
ask_local_ai("Compare the average price_realistic of the iPhone 14 between London and New York. Filter WHERE product_name LIKE '%iPhone 14%' AND city IN ('London', 'New York'). GROUP BY city.")
ask_local_ai("What was the most expensive item sold at the 'Apple Covent Garden' store? Show product_name, price_realistic. Filter WHERE store_name = 'Apple Covent Garden'. ORDER BY price_realistic DESC LIMIT 1.")



💬  Compare the average price_realistic of the iPhone 14 between London and New York. Filter WHERE product_name LIKE '%iPhone 14%' AND city IN ('London', 'New York'). GROUP BY city.

⚙  SQL: SELECT AVG(price_realistic) AS avg_price, city 
FROM sales 
WHERE product_name LIKE '%iPhone 14%' AND city IN ('London', 'New York') 
GROUP BY city LIMIT 50;

🔍  Raw Data: [(951.4505082835534, 'New York'), (937.5550820231608, 'London')]

📊  Analyst Briefing:
   ### Executive Briefing

#### Key Insight:
The average price_realistic of the iPhone 14 is significantly higher in New York compared to London.

**Supporting Details:**
- **New York:** The average price_realistic for an iPhone 14 is $951.45.
- **London:** The average price_realistic for an iPhone 14 is $937.56.

#### Business Context:
This difference in pricing could be attributed to various factors such as local supply chain dynamics, market demand, and economic conditions specific to each city. Understanding these differences can help Apple

In [19]:
ask_local_ai("Are there any MacBooks available that have an average price under $1500 in 2024?")


💬  Are there any MacBooks available that have an average price under $1500 in 2024?

⚙  SQL: SELECT product_name, AVG(price_realistic) AS avg_price
FROM sales
WHERE category_name = 'Mac'
  AND year = 2024
GROUP BY product_name
HAVING AVG(price_realistic) < 1500 LIMIT 50;

🔍  Raw Data: 

📊  Analyst Briefing:
   Certainly. Based on the provided data, as of 2024, there are no MacBook models with an average price under $1500 available. The current market suggests that most MacBooks, including those in the lower price range, have prices exceeding this threshold. This trend aligns with Apple's strategy to maintain high-quality and premium pricing for their products, which often includes advanced features and performance optimizations.

To address this trend, Apple could consider offering more affordable options or exploring alternative product lines within its portfolio that might be more accessible to a broader audience while still maintaining quality standards.



In [20]:
ask_local_ai("Are there any iphone available that have an average price under $1500 in 2024?")


💬  Are there any iphone available that have an average price under $1500 in 2024?

⚙  SQL: SELECT product_name, AVG(price_realistic) AS avg_price
FROM sales
WHERE product_name LIKE '%iPhone%'
AND year = 2024
GROUP BY product_name
HAVING AVG(price_realistic) <= 1500 LIMIT 50;

🔍  Raw Data: [('iPhone 15 Pro Max', 1156.9492150413903), ('iPhone 15 Plus', 865.1005757919226), ('iPhone 14 Plus', 795.2319677824346), ('iPhone 16 Pro', 1025.6690418409564), ('iPhone 13 mini', 561.3356753844282), ...

📊  Analyst Briefing:
   Yes, there are several iPhone models available with an average price under $1500 in 2024. The top three options are:

1. **iPhone SE (3rd Gen)** - Average Price: $379.90

2. **iPhone 14 Pro Max** - Average Price: $971.87

3. **iPhone 15 Pro Max** - Average Price: $1,230.99

These models offer good value for money and are well-suited for various price-conscious consumers looking to upgrade their iPhone lineup.



In [21]:
ask_local_ai("What is the absolute cheapest product I can buy from the 'Accessories' category?")


💬  What is the absolute cheapest product I can buy from the 'Accessories' category?

⚙  SQL: SELECT product_name, AVG(price_realistic) AS average_price
FROM sales
WHERE category_name = 'Accessories'
GROUP BY product_name
HAVING AVG(price_realistic) <= (SELECT MIN(price_realistic) FROM sales WHERE category_name = 'Accessories')
ORDER BY average_price ASC
LIMIT 1;

🔍  Raw Data: 

📊  Analyst Briefing:
   **Executive Briefing**

**Key Insight:** The absolute cheapest product in the 'Accessories' category is the **Apple AirPods Pro**, priced at **$249.00 USD**.

**Supporting Details:**
- This product offers high-quality audio and wireless connectivity, making it a popular choice for users.
- It has received positive reviews for its durability and performance in various environments.
- The price is competitive with other premium headphones in the market, offering excellent value for money.

**Business Context:** Apple's focus on innovation and quality ensures that their products are not onl

In [22]:
# Year-over-Year Price Comparison — Rule 15 ensures conditional aggregation
ask_local_ai("Compare the average price_realistic of the iPhone 13 in 2023 vs 2024. Use: AVG(CASE WHEN year=2023 THEN price_realistic END) AS price_2023, AVG(CASE WHEN year=2024 THEN price_realistic END) AS price_2024. Filter WHERE product_name LIKE '%iPhone 13%' AND year IN (2023, 2024).")



💬  Compare the average price_realistic of the iPhone 13 in 2023 vs 2024. Use: AVG(CASE WHEN year=2023 THEN price_realistic END) AS price_2023, AVG(CASE WHEN year=2024 THEN price_realistic END) AS price_2024. Filter WHERE product_name LIKE '%iPhone 13%' AND year IN (2023, 2024).

⚙  SQL: SELECT 
    AVG(CASE WHEN year = 2023 THEN price_realistic END) AS price_2023,
    AVG(CASE WHEN year = 2024 THEN price_realistic END) AS price_2024
FROM 
    sales
WHERE 
    product_name LIKE '%iPhone 13%' AND year IN (2023, 2024) LIMIT 50;

🔍  Raw Data: [(805.5495061122996, 718.2750624898816)]

📊  Analyst Briefing:
   ### Executive Briefing on iPhone 13 Price Comparison

#### Key Insight:
The average price_realistic for the iPhone 13 in 2023 was $805.55, while in 2024 it decreased to $718.28.

#### Supporting Details:
- **Year 2023:** The iPhone 13 had an average price of $805.55.
- **Year 2024:** The average price dropped to $718.28, representing a decrease of approximately 12% compared to the prev

In [23]:
ask_local_ai("Which month in 2024 had the cheapest average realistic price for the ipad?")


💬  Which month in 2024 had the cheapest average realistic price for the ipad?

⚙  SQL: SELECT month, AVG(price_realistic) AS avg_price
FROM sales
WHERE product_name LIKE '%iPad%' AND year = 2024
GROUP BY month
ORDER BY avg_price ASC
LIMIT 1;

🔍  Raw Data: [(11, 740.952282146336)]

📊  Analyst Briefing:
   **Executive Briefing**

Good morning, Apple executives.

I am pleased to present our findings on the cheapest average realistic price of an iPad in 2024. Based on the data provided, November was the month with the lowest average realistic price for an iPad, at $740.95.

This trend is significant as it indicates that consumers may be more inclined to purchase iPads during this period, potentially leading to increased sales and revenue for Apple. The low price point suggests that Apple has been able to maintain its competitive edge in the market while also offering a value proposition that resonates with customers.

To further support this insight, we have observed a slight increase in 

In [24]:
# Specific Product Comparison by Year — Rule 12 ensures exact filtering
ask_local_ai("What is the average price_realistic of iPhone 13 and iPhone 14 for each year? Filter WHERE product_name LIKE '%iPhone 13%' OR product_name LIKE '%iPhone 14%'. GROUP BY product_name, year. ORDER BY product_name, year.")



💬  What is the average price_realistic of iPhone 13 and iPhone 14 for each year? Filter WHERE product_name LIKE '%iPhone 13%' OR product_name LIKE '%iPhone 14%'. GROUP BY product_name, year. ORDER BY product_name, year.

⚙  SQL: SELECT 
    product_name,
    AVG(price_realistic) AS average_price,
    SUM(CASE WHEN year = 2023 THEN price_realistic END) AS price_2023,
    SUM(CASE WHEN year = 2024 THEN price_realistic END) AS price_2024
FROM 
    sales
WHERE 
    product_name LIKE '%iPhone 13%' OR product_name LIKE '%iPhone 14%'
GROUP BY 
    product_name, year
ORDER BY 
    product_name, year LIMIT 50;

🔍  Raw Data: [('iPhone 13', 791.6215596360211, None, None), ('iPhone 13', 756.0492524136782, None, None), ('iPhone 13', 718.2856305294363, 995543.8839137986, None), ('iPhone 13', 646.7346167649285, None, 184319.36...

📊  Analyst Briefing:
   Thank you for sharing the data with me. Based on the provided query and results, here are the key insights:

### Average Price Realistic for iPhone

In [25]:
ask_local_ai("top 10 product for each category in 2023")


💬  top 10 product for each category in 2023

⚙  SQL: SELECT 
    category_name, 
    product_name, 
    SUM(quantity_realistic) AS total_quantity_2023
FROM 
    sales
WHERE 
    year = 2023
GROUP BY 
    category_name, 
    product_name
ORDER BY 
    total_quantity_2023 DESC
LIMIT 10;

🔍  Raw Data: [('Audio', 'HomePod (2nd Gen)', 109284), ('Wearable', 'Apple Watch Series 9', 107547), ('Smartphone', 'iPhone 15', 96818), ('Smartphone', 'iPhone 15 Plus', 89826), ('Smartphone', 'iPhone 15 Pro', 8246...

📊  Analyst Briefing:
   **Executive Briefing**

Good morning, Apple Executives. Today, I'm presenting the top 10 products for each category in 2023.

### Key Insight:
The top 10 products across all categories demonstrate a strong preference for Apple's flagship devices and subscription services. This indicates a robust consumer base that values quality and innovation from Apple.

### Detailed Findings:

- **Audio**: The HomePod (2nd Gen) remains the top product in this category, with 109,2

In [29]:
ask_local_ai("top 10 product for each category in 2023")


💬  top 10 product for each category in 2023

⚙  SQL: SELECT 
    category_name, 
    product_name, 
    SUM(quantity_realistic) AS total_quantity_2023
FROM 
    sales
WHERE 
    year = 2023
GROUP BY 
    category_name, 
    product_name
ORDER BY 
    total_quantity_2023 DESC
LIMIT 10;

🔍  Raw Data: [('Audio', 'HomePod (2nd Gen)', 109284), ('Wearable', 'Apple Watch Series 9', 107547), ('Smartphone', 'iPhone 15', 96818), ('Smartphone', 'iPhone 15 Plus', 89826), ('Smartphone', 'iPhone 15 Pro', 8246...

📊  Analyst Briefing:
   **Executive Briefing on Top Products for Each Category in 2023**

**Key Insight:** Apple's top products across various categories have been a consistent focus, with the iPhone series leading the smartphone category. The HomePod (2nd Gen) has maintained its position as the top product in the Audio category.

**Supporting Details:**
- **Audio:** HomePod (2nd Gen) remains the top product with 109,284 units sold.
- **Wearable:** Apple Watch Series 9 is the leading weara

In [32]:
ask_local_ai("What was the total revenue generated for MacBooks in the United States grouped by month in 2024? Sort it from January to December.")


💬  What was the total revenue generated for MacBooks in the United States grouped by month in 2024? Sort it from January to December.

⚙  SQL: SELECT 
    SUM(sales_amount_realistic) AS total_revenue,
    month
FROM 
    sales
WHERE 
    product_name LIKE '%MacBook%' AND country_norm_mapped = 'USA' AND year = 2024
GROUP BY 
    month
ORDER BY 
    month LIMIT 50;

🔍  Raw Data: 

📊  Analyst Briefing:
   Executive Briefing:

**Key Insight:** The total revenue generated for MacBooks in the United States from January to December 2024 was $1.5 billion.

**Supporting Details:**
- **January:** $300 million
- **February:** $350 million
- **March:** $400 million
- **April:** $450 million
- **May:** $500 million
- **June:** $550 million
- **July:** $600 million
- **August:** $650 million
- **September:** $700 million
- **October:** $750 million
- **November:** $800 million
- **December:** $850 million

**Business Context:**
This trend indicates a consistent growth in MacBook sales and revenue t

In [30]:
ask_local_ai("What was the total revenue generated for MacBooks in the United States grouped by month in 2024? Sort it from January to December.")


💬  What was the total revenue generated for MacBooks in the United States grouped by month in 2024? Sort it from January to December.

⚙  SQL: SELECT 
    SUM(sales_amount_realistic) AS total_revenue,
    month
FROM 
    sales
WHERE 
    product_name LIKE '%MacBook%' AND country_norm_mapped = 'USA' AND year = 2024
GROUP BY 
    month
ORDER BY 
    month LIMIT 50;

🔍  Raw Data: 

📊  Analyst Briefing:
   **Executive Briefing**

**Key Insight:** The total revenue generated for MacBooks in the United States from January to December 2024 was $1.5 billion.

**Supporting Details:**
- **January:** $300 million
- **February:** $350 million
- **March:** $400 million
- **April:** $450 million
- **May:** $500 million
- **June:** $550 million
- **July:** $600 million
- **August:** $650 million
- **September:** $700 million
- **October:** $750 million
- **November:** $800 million
- **December:** $850 million

**Business Context:** This trend indicates a steady increase in MacBook sales and revenue 